In [ ]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import pandas as pd
from copy import deepcopy
import h5py
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
DATASET = "A4"  # or ADNI

PROJECT_DIR = Path(os.environ["PROJECT_DIR"])
ATLAS_DIR = Path(os.environ["ATLAS_DIR"])
VISUAL_READS_DIR = Path(os.environ["VISUAL_READS_DIR"])
DATA_DIR = Path(os.environ["DATA_DIR"])
META_FILE = Path(os.environ["META_FILE"])
MAEMI_OUTPUTS_DIR = Path(os.environ["MAEMI_OUTPUTS_DIR"])

df = pd.read_csv(META_FILE)
visual_reads_csv = Path(VISUAL_READS_DIR, "visual_reads.csv")
df_visual = pd.read_csv(visual_reads_csv)

if DATASET == "A4":
    df = df[
        ~df["BIDS_ID"].astype(str).str.contains("LEARN", case=False, na=False)
    ].copy()
    df_visual = df_visual[
            ~df_visual["ID"].astype(str).str.contains("LEARN", case=False, na=False)
        ].copy()

### Resample & Connected Components Functions

In [ ]:
def resample_img(itk_image, spacing, is_label=False):

    original_spacing = itk_image.GetSpacing()
    original_size = itk_image.GetSize()

    out_size = [
        int(np.round(original_size[0] * (original_spacing[0] / spacing[0]))),
        int(np.round(original_size[1] * (original_spacing[1] / spacing[1]))),
        int(np.round(original_size[2] * (original_spacing[2] / spacing[2]))),
    ]

    resample = sitk.ResampleImageFilter()
    resample.SetOutputSpacing(spacing)
    resample.SetSize(out_size)
    resample.SetOutputDirection(itk_image.GetDirection())
    resample.SetOutputOrigin(itk_image.GetOrigin())
    resample.SetTransform(sitk.Transform())
    if is_label:
        resample.SetDefaultPixelValue(0)
        resample.SetInterpolator(sitk.sitkNearestNeighbor)
    else:
        resample.SetDefaultPixelValue(itk_image.GetPixelIDValue())
        resample.SetInterpolator(sitk.sitkBSpline)

    return resample.Execute(itk_image)


def read_nii_file(path, orientation=None):
    data = sitk.ReadImage(path)
    if orientation is not None:
        data = sitk.DICOMOrient(data, orientation)
    return data


def get_itk_from_path(path, orientation=None):
    path = Path(path)
    return read_nii_file(path, orientation)


def get_array_from_itk(image):
    data = sitk.GetArrayFromImage(image)
    if len(data.shape) != 3:
        data = data.squeeze()
    data = np.transpose(data, (2, 1, 0))
    return data

In [ ]:
def characterise_connected_components(img_data, connectivity=26, statistic="countCC"):
    """
    Adapted from jadenecke mrpipe/Toolboxes/submodules/custom/CCStats.py
    """
    import numpy as np
    from scipy.ndimage import label
    from skimage.measure import regionprops
    import scipy.ndimage as ndi

    voxel_volume = 1.5 * 1.5 * 1.5 

    # Convert connectivity parameter to structure
    if connectivity == 6:  # Only face-adjacent voxels are neighbors (no diagonals).
        structure = ndi.generate_binary_structure(3, 1)  # 6-connectivity
    elif connectivity == 18:  # Face- and edge-adjacent voxels are neighbors
        structure = ndi.generate_binary_structure(3, 2)  # 18-connectivity
    elif connectivity == 26:  # All voxels in a 3×3×3 cube around the center, including corners, are neighbors
        structure = ndi.generate_binary_structure(3, 3)  # 26-connectivity
    else:
        # Default to 26-connectivity if invalid value provided
        structure = ndi.generate_binary_structure(3, 3)

    # Label connected components
    labeled_mask, num_cc = label(img_data, structure=structure)

    # If no connected components found
    if num_cc == 0:
        return 0 if statistic in ["countCC", "totalVoxel", "totalVolume", "maxVolume"] else "NA"

    # Calculate statistics
    regions = regionprops(labeled_mask)
    cc_sizesVoxel = [region.area for region in regions]
    cc_sizesVolume = [size * voxel_volume for size in cc_sizesVoxel]

    # Calculate requested statistic
    if statistic == "countCC":
        result = num_cc
    elif statistic == "minVoxel":
        result = np.min(cc_sizesVoxel)
    elif statistic == "maxVoxel":
        result = np.max(cc_sizesVoxel)
    elif statistic == "meanVoxel":
        result = np.mean(cc_sizesVoxel)
    elif statistic == "stdVoxel":
        result = np.std(cc_sizesVoxel)
    elif statistic == "totalVoxel":
        result = np.sum(cc_sizesVoxel)
    elif statistic == "minVolume":
        result = np.min(cc_sizesVolume)
    elif statistic == "maxVolume":
        result = np.max(cc_sizesVolume)
    elif statistic == "meanVolume":
        result = np.mean(cc_sizesVolume)
    elif statistic == "stdVolume":
        result = np.std(cc_sizesVolume)
    elif statistic == "totalVolume":
        result = np.sum(cc_sizesVolume)
    else:
        # Default to count if invalid statistic
        result = num_cc

    return result

In [ ]:
to_resample = ["TAUVID_Labels_corticalGM_Temporal10_dil2mm.nii.gz", "TAUVID_Labels_corticalGM_Temporal10.nii.gz"]

for file in to_resample:
    filepath = Path(VISUAL_READS_DIR, file)
    file_out = filepath.parent / filepath.stem.replace(".nii", "_1_5mm.nii.gz")

    ds = get_itk_from_path(filepath)
    ds = resample_img(ds, [1.5, 1.5, 1.5], is_label=True)

    ds = ds[4:117, 4:141, :113]  # To crop Schaefer atlas to right size after resampling

### Split (entire) data set into folds - ADNI only

In [ ]:
num_folds = 5

df_with_folds = df.copy()

if DATASET == "ADNI":
    # Assign one representative disease/biomarker state per participant
    severity_order = {
        'CN_amyNeg_tauNeg': 0,
        'CN_amyPos_tauNeg': 1,
        'CN_amyNeg_tauPos': 2,
        'CN_amyPos_tauPos': 3,
        'MCI_amyNeg_tauNeg': 4,
        'MCI_amyPos_tauNeg': 5,
        'MCI_amyPos_tauPos': 6,
        'AD_amyNeg_tauNeg': 7,
        'AD_amyPos_tauNeg': 8,
        'AD_amyPos_tauPos': 9
    }

    patient_labels = df.groupby('BIDS_ID')['dx_granular'].apply(
        lambda stages: max(stages, key=lambda s: severity_order[s])
    ).reset_index()

    stratify_col = 'dx_granular'


elif DATASET == "A4":
    # A4 participants are CN, so stratify by biomarker status instead
    # Use the existing tau/amyloid columns in the A4 dataframe
    patient_labels = df.groupby('BIDS_ID').first().reset_index()

    patient_labels['biomarker_group'] = (
        patient_labels['amy'].astype(str)
        + '_'
        + patient_labels['tau'].astype(str)
    )

    stratify_col = 'biomarker_group'


# Participant-level splitting
X = patient_labels['BIDS_ID']
y = patient_labels[stratify_col]
groups = patient_labels['BIDS_ID']

skf = StratifiedGroupKFold(
    n_splits=num_folds,
    shuffle=True,
    random_state=42
)

fold_assignments = {}

for fold, (_, val_idx) in enumerate(skf.split(X, y, groups)):
    val_patients = patient_labels.iloc[val_idx]['BIDS_ID']

    for pid in val_patients:
        fold_assignments[pid] = fold


# Convert fold assignments to DataFrame
fold_df = pd.DataFrame(
    list(fold_assignments.items()),
    columns=['BIDS_ID', 'fold']
)

# Merge back into full DATASET
df_with_folds = df.merge(fold_df, on='BIDS_ID')

### Get tau status for different intensity and cluster size thresholds

In [ ]:
tau_thresholds = [1.3, 1.4, 1.5, 1.65, 1.8, 2.0, 2.25]  # Tau SUVR thresholds
anomaly_thresholds = [0.01, 0.05, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]  # Anomaly map thresholds
cluster_size_thresholds = [0, 50, 75, 100, 150, 200, 250, 300, 350]  # Thresholds for cluster size to consider as tau positive
tauvid_thresholds = {}

if DATASET == "ADNI":
    pred_path = Path(MAEMI_OUTPUTS_DIR, "pred_linkfile.h5")
    df_test = df[["identifier", "dx_granular"]].copy()
    df_test["tau"] = df_test["dx_granular"].str.split('_').str[-1]
    df_test["dx"] = df_test["dx_granular"]
    pids = df_test["identifier"]
elif DATASET == "A4":
    pred_path = Path(MAEMI_OUTPUTS_DIR, "preds.h5")
    df_test = df[["identifier", "tau"]].copy()
    df_test = df[["identifier", "tau"]].copy()
    df_test["dx"] = "CN"
    pids = df_test.loc[df_test["identifier"].str.contains("A4", na=False), "identifier"]

orig_imgs_path = Path(PROJECT_DIR, "tauPET/no_norm_patches.h5")
df_test["visual_read"] = np.nan

# Get only PLT regions bilaterally
labels = [1, 3, 11, 5, 6, 16]  # only PLT region.
tauvid_mask_file = Path(VISUAL_READS_DIR, "TAUVID_Labels_corticalGM_Temporal10_dil2mm_1_5mm.nii.gz")
tauvid_mask = get_itk_from_path(tauvid_mask_file)
tauvid_mask = get_array_from_itk(tauvid_mask)

# whole cerebellum mask
wholecer_file = Path(ATLAS_DIR, "lpba40_resampled_1_5mm_pad.nii")
wholecer_mask = get_itk_from_path(wholecer_file)
wholecer_mask = get_array_from_itk(wholecer_mask)
wholecer_mask = wholecer_mask == 181
largest_cer_slice = np.argmax(wholecer_mask.sum(axis=(0, 1)))
largest_cer_slice_mask = np.zeros_like(wholecer_mask)
largest_cer_slice_mask[:, :, largest_cer_slice] = wholecer_mask[:, :, largest_cer_slice]

cc_stats_tau = {threshold: {} for threshold in tau_thresholds}
cc_stats_anomaly = {threshold: {} for threshold in anomaly_thresholds}

no_visual_read = []
more_than_one_visual_read = []
with h5py.File(pred_path, "r") as f, h5py.File(orig_imgs_path, "r") as orig_f:
    for pid_ind, (pid, dx) in enumerate(zip(pids, df_test["dx"])):
        print("-"*40)
        print(f"[{pid_ind+1}/{len(pids)}]: {pid}, {dx}")
        plt_mask = np.isin(tauvid_mask, labels).astype(np.int8)

        # Get visual read
        if DATASET == "ADNI":
            id = pid.split("sub-")[1].split("_")[0].lstrip("0")
            df_visual_id = df_visual[df_visual["ID"] == id].copy()

            # ADNI can have multiple scans/visual reads, so match by tau-PET date
            df_visual_id["date_tau_PET"] = pd.to_datetime(
                df_visual_id["date_tau_PET"],
                format="%d-%b-%Y"
            )

            examdate_tau = pd.to_datetime(
                pid.split("_tau")[1].split("_")[0],
                format="%Y-%m-%d"
            )

            visual_row = df_visual_id[
                df_visual_id["date_tau_PET"] == examdate_tau
            ]

        elif DATASET == "A4":
            id = pid.split("sub-A4")[1].split("_tau")[0]
            df_visual_id = df_visual[df_visual["ID"] == id].copy()

            # A4 has one visual-read entry per subject, so no date matching needed
            visual_row = df_visual_id


        # Check that a visual read was found
        if visual_row.shape[0] == 0:
            no_visual_read.append(pid)
            df_test.drop(
                df_test[df_test["identifier"] == pid].index,
                inplace=True
            )
            continue

        if visual_row.shape[0] > 1:
            more_than_one_visual_read.append(pid)
            print(f"More than one matching visual read for {pid}")
            continue


        # Get visual read status
        visual_read = visual_row["visual_read_binary"].iloc[0]

        if visual_read == 0:
            df_test.loc[df_test["identifier"] == pid, "visual_read"] = "tauNeg"
        elif visual_read == 1:
            df_test.loc[df_test["identifier"] == pid, "visual_read"] = "tauPos"
        else:
            print(f"Unexpected visual read value: {visual_read} for {pid}")
            df_test.drop(
                df_test[df_test["identifier"] == pid].index,
                inplace=True
            )
            continue


        # Get diagnosis and amyloid status
        dx = visual_row["dx"].iloc[0]
        abeta = visual_row["amyloid_status"].iloc[0]

        if abeta == 0:
            df_test.loc[df_test["identifier"] == pid, "dx"] = f"{dx}_amyNeg"
        elif abeta == 1:
            df_test.loc[df_test["identifier"] == pid, "dx"] = f"{dx}_amyPos"

        # Get individual GM mask
        if "dil" in str(tauvid_mask_file):
            mask_file = str(df[df["identifier"] == pid]["anat_mask"].values[0])
            mask_sitk = read_nii_file(mask_file)
            mask_array = get_array_from_itk(mask_sitk)
            mask_array = np.round(mask_array)
            mask_array = mask_array.squeeze()
            GM_mask = (mask_array == 2).astype(int)
            plt_mask = plt_mask * GM_mask

        # Tau SUVR ----------------------------------------------
        img = orig_f[pid]["tauPET"][()]
        img = img.squeeze()

        # Since already normalised by inf cer, just use threshold 1.3
        # I.e. don't using 1.65 of the cerebellum
        img[plt_mask == 0] = np.nan
        for threshold in tau_thresholds:
            tau_thresholded = (img > threshold).astype(np.int8)
            cc_stats_tau[threshold][pid] = characterise_connected_components(tau_thresholded, connectivity=26, statistic="maxVolume")

        # Anomaly map -------------------------------------------
        anomaly_map = f[pid]["diff"][()]
        anomaly_map = anomaly_map.squeeze()
        anomaly_map[plt_mask == 0] = np.nan

        for threshold in anomaly_thresholds:
            anomaly_thresholded = (anomaly_map > threshold).astype(np.int8)
            cc_stats_anomaly[threshold][pid] = characterise_connected_components(anomaly_thresholded, connectivity=26, statistic="maxVolume")

print(f"More than one visual read for: {more_than_one_visual_read}")
print("-"*40)
print(f"No visual read for: {no_visual_read}")
print(len(no_visual_read))

tau_status_tau = {cluster_size:[] for cluster_size in cluster_size_thresholds}
tau_status_anomaly = {cluster_size: [] for cluster_size in cluster_size_thresholds}

df_tau = pd.DataFrame.from_dict(cc_stats_tau, orient="index").T
df_anomaly = pd.DataFrame.from_dict(cc_stats_anomaly, orient="index").T

for cluster_size in cluster_size_thresholds:
    tau_status_tau[cluster_size] = pd.DataFrame(
        np.where(df_tau > cluster_size, "tauPos", "tauNeg"),
        columns=df_tau.columns,
        index=df_tau.index
    )

    tau_status_anomaly[cluster_size] = pd.DataFrame(
        np.where(df_anomaly > cluster_size, "tauPos", "tauNeg"),
        columns=df_anomaly.columns,
        index=df_anomaly.index
    )

### Get population values

In [ ]:
GT = "visual_read"  # or "schaefer"
if GT == "visual_read":
    GT_column = "visual_read"
else:
    GT_column = "tau"
tau_GT = df_test[["identifier", GT_column]].set_index("identifier")
tau_GT.index.name = None
num_tauPos_GT = tau_GT[GT_column].str.count("tauPos").sum()
tauvid_threshold = 4000

num_tauPos_tau = {cluster_size: [] for cluster_size in cluster_size_thresholds}
num_tauPos_anomaly = {cluster_size: [] for cluster_size in cluster_size_thresholds}

# Agreement with tau positivity in Nico's spreadsheet
perc_agreement_tau = {cluster_size: [] for cluster_size in cluster_size_thresholds}
perc_agreement_anomaly = {cluster_size: [] for cluster_size in cluster_size_thresholds}
# perc_agreement_anomaly_tauvid = {cluster_size: [] for cluster_size in cluster_size_thresholds}
test_tau = {cluster_size: [] for cluster_size in cluster_size_thresholds}
test_anomaly = {cluster_size: [] for cluster_size in cluster_size_thresholds}

for cluster_size in cluster_size_thresholds:
    num_tauPos_tau[cluster_size] = tau_status_tau[cluster_size].apply(lambda x: (x == "tauPos").sum(), axis=0)
    num_tauPos_anomaly[cluster_size] = tau_status_anomaly[cluster_size].apply(lambda x: (x == "tauPos").sum(), axis=0)

    agreement_tau = tau_status_tau[cluster_size].eq(tau_GT[GT_column], axis=0)
    perc_agreement_tau[cluster_size] = agreement_tau.mean() * 100
    agreement_anomaly = tau_status_anomaly[cluster_size].eq(tau_GT[GT_column], axis=0)
    perc_agreement_anomaly[cluster_size] = agreement_anomaly.mean() * 100

    test_tau[cluster_size] = num_tauPos_tau[cluster_size] / num_tauPos_GT
    test_anomaly[cluster_size] = num_tauPos_anomaly[cluster_size] / num_tauPos_GT

num_tauPos_tau = pd.DataFrame(num_tauPos_tau)
num_tauPos_anomaly = pd.DataFrame(num_tauPos_anomaly)
test_tau = pd.DataFrame(test_tau)
test_anomaly = pd.DataFrame(test_anomaly)
perc_agreement_tau = pd.DataFrame(perc_agreement_tau)
perc_agreement_anomaly = pd.DataFrame(perc_agreement_anomaly)

In [ ]:
if GT == "visual_read":
    agreement_title = "Visual Read"
else:
    agreement_title = "Schaefer Global SUVR"
modalities = ["Tau SUVR image", "Anomaly Map", "Anomaly Map"]
agreements = [agreement_title, agreement_title, f"TauVID cluster size {tauvid_threshold}"]
# Plot the heatmap
for data, modality in zip([num_tauPos_tau, num_tauPos_anomaly], modalities):
    plt.figure(figsize=(8, 6), dpi=100)
    sns.heatmap(data.T, annot=True, fmt="d", cmap="YlOrRd")
    plt.ylabel("Cluster Size Threshold")
    plt.xlabel("Intensity Threshold")
    plt.title(f"{modality}: Num Tau Positive Subjects")
    plt.show()
    
for data, modality, agreement in zip([perc_agreement_tau, perc_agreement_anomaly], modalities, agreements):
    plt.figure(figsize=(8, 6), dpi=100)
    sns.heatmap(data.T, annot=True, fmt=".2f", cmap="YlOrRd")
    plt.ylabel("Cluster Size (mm3) Threshold")
    plt.xlabel("Intensity Threshold")
    plt.title(f"{modality}: % agreement with tau status from {agreement}")
    plt.show()

In [ ]:
tau_clusterthr = 200
tau_intensitythr = 1.65
anomaly_clusterthr = 200
anomaly_intensitythr = 0.08

agreement_both_methods = tau_status_tau[tau_clusterthr][tau_intensitythr].eq(tau_status_anomaly[anomaly_clusterthr][anomaly_intensitythr].values, axis=0)
disagree_pids = agreement_both_methods.index[agreement_both_methods == False].tolist()
print(disagree_pids)

In [ ]:
diagnosis = "AlzCS dem_amyPos"  # AlzCS dem_amyPos
target_names = ["tauNeg", "tauPos"]

if diagnosis == "all":
    y_true = tau_GT['visual_read'].values
    y_pred_tau = tau_status_tau[tau_clusterthr][tau_intensitythr].values
    y_pred_anomaly = tau_status_anomaly[anomaly_clusterthr][anomaly_intensitythr].values
else:
    diagnoses = df_test[["identifier", "dx"]].set_index("identifier")
    indices = diagnoses[diagnoses["dx"] == diagnosis].index
    y_true = tau_GT.loc[indices, "visual_read"].values
    y_pred_tau = tau_status_tau[tau_clusterthr][tau_intensitythr].loc[indices].values
    y_pred_anomaly = tau_status_anomaly[anomaly_clusterthr][anomaly_intensitythr].loc[indices].values
cm_tau = confusion_matrix(y_true, y_pred_tau, labels=target_names)
cm_anomaly = confusion_matrix(y_true, y_pred_anomaly, labels=target_names)

plt.figure(figsize=(6, 5), dpi=150)
sns.heatmap(cm_tau, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.title(f"Confusion Matrix for Tau SUVR (Cluster Size {tau_clusterthr}, Intensity {tau_intensitythr})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

plt.figure(figsize=(6, 5), dpi=150)
sns.heatmap(cm_anomaly, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.title(f"Confusion Matrix for Anomaly Map (Cluster Size {anomaly_clusterthr}, Intensity {anomaly_intensitythr})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


# For binary classification:
print('Tau SUVR:\n', classification_report(y_true, y_pred_tau, target_names=target_names))
print(f'Balanced accuracy: {balanced_accuracy_score(y_true, y_pred_tau):.3f}', )
print('Anomaly map:\n', classification_report(y_true, y_pred_anomaly, target_names=target_names))
print(f'Balanced accuracy: {balanced_accuracy_score(y_true, y_pred_anomaly):.3f}', )



### k-fold cross validation 

In [ ]:
diagnosis = "all"  # ["all", "CU_amyPos", "MCI_amyPos", "AlzCS dem_amyPos"]
target_names = ["tauNeg", "tauPos"]
GT_column = "visual_read"

diagnoses = df_test[["identifier", "dx"]].set_index("identifier")
tau_GT = df_test[["identifier", GT_column]].set_index("identifier")
tau_GT.index.name = None

best_thresholds_tau = {
    "cluster_size": {
        k: -1 for k in range(num_folds)
    },
    "intensity": {
        k: -1 for k in range(num_folds)
    }
}
best_thresholds_anomaly = deepcopy(best_thresholds_tau)
reports_tau_allfolds = []
reports_anomaly_allfolds = []
fold = 0

# Store held-out predictions across all folds
y_true_allfolds = []
y_pred_tau_allfolds = []
y_pred_anomaly_allfolds = []
identifier_allfolds = []
fold_allfolds = []

for fold in range(num_folds):
    # -----------------------------------------------------
    # Find best thresholds based on training set
    training_inds = df_with_folds[df_with_folds["fold"] != fold]["identifier"].values

    print(f"\nFold {fold}")
    print("training_inds:", len(training_inds))
    print("GT available:", tau_GT.index.isin(training_inds).sum())
    print(
        "SUVR available:",
        tau_status_tau[cluster_size_thresholds[0]].index.isin(training_inds).sum()
    )
    print(
        "Anomaly available:",
        tau_status_anomaly[cluster_size_thresholds[0]].index.isin(training_inds).sum()
    )

    print(
        "GT but no SUVR:",
        tau_GT.index[
            tau_GT.index.isin(training_inds)
            & ~tau_GT.index.isin(tau_status_tau[cluster_size_thresholds[0]].index)
        ].tolist()
    )

    # Agreement with visual reads
    balanced_acc_tau = {
        cluster_size: []
        for cluster_size in cluster_size_thresholds
    }

    balanced_acc_anomaly = {
        cluster_size: []
        for cluster_size in cluster_size_thresholds
    }

    for cluster_size in cluster_size_thresholds:

        # Get predictions for this cluster-size threshold
        tau_status_tau_train = tau_status_tau[cluster_size]
        tau_status_anomaly_train = tau_status_anomaly[cluster_size]

        # Use only training observations that exist in:
        #   1. ground truth
        #   2. SUVR predictions
        #   3. anomaly predictions
        common_training_inds = (
            tau_GT.index
            .intersection(tau_status_tau_train.index)
            .intersection(tau_status_anomaly_train.index)
        )

        common_training_inds = common_training_inds[
            common_training_inds.isin(training_inds)
        ]

        # Explicitly select the SAME observations, in the SAME order
        tau_GT_train = tau_GT.loc[common_training_inds]
        tau_status_tau_train = tau_status_tau_train.loc[common_training_inds]
        tau_status_anomaly_train = tau_status_anomaly_train.loc[common_training_inds]

        for intensity_threshold in tau_status_tau_train.columns:
            balanced_acc_tau[cluster_size].append(
                balanced_accuracy_score(
                    tau_GT_train[GT_column],
                    tau_status_tau_train[intensity_threshold]
                )
            )

        for intensity_threshold in tau_status_anomaly_train.columns:
            balanced_acc_anomaly[cluster_size].append(
                balanced_accuracy_score(
                    tau_GT_train[GT_column],
                    tau_status_anomaly_train[intensity_threshold]
                )
            )

    balanced_acc_tau = pd.DataFrame(
        balanced_acc_tau,
        index=tau_status_tau_train.columns
    )

    balanced_acc_anomaly = pd.DataFrame(
        balanced_acc_anomaly,
        index=tau_status_anomaly_train.columns
    )

    # -----------------------------------------------------
    # Best anomaly thresholds
    idx = balanced_acc_anomaly.values.argmax()
    row, col = np.unravel_index(
        idx,
        balanced_acc_anomaly.shape
    )

    anomaly_intensitythr = balanced_acc_anomaly.index[row]
    anomaly_clusterthr = balanced_acc_anomaly.columns[col]

    best_thresholds_anomaly["cluster_size"][fold] = anomaly_clusterthr
    best_thresholds_anomaly["intensity"][fold] = anomaly_intensitythr


    # Best SUVR thresholds
    idx = balanced_acc_tau.values.argmax()
    row, col = np.unravel_index(
        idx,
        balanced_acc_tau.shape
    )

    tau_intensitythr = balanced_acc_tau.index[row]
    tau_clusterthr = balanced_acc_tau.columns[col]

    best_thresholds_tau["cluster_size"][fold] = tau_clusterthr
    best_thresholds_tau["intensity"][fold] = tau_intensitythr

    test_inds = df_with_folds[df_with_folds["fold"] == fold]["identifier"].values
    tau_GT_test = tau_GT.loc[tau_GT.index.isin(test_inds)]
    y_pred_tau_test = tau_status_tau[tau_clusterthr][tau_intensitythr].loc[tau_status_tau[tau_clusterthr][tau_intensitythr].index.isin(test_inds)]
    y_pred_anomaly_test = tau_status_anomaly[anomaly_clusterthr][anomaly_intensitythr].loc[tau_status_anomaly[anomaly_clusterthr][anomaly_intensitythr].index.isin(test_inds)]

    if diagnosis == "all":
        y_true = tau_GT_test['visual_read'].values
        y_pred_tau = y_pred_tau_test.values
        y_pred_anomaly = y_pred_anomaly_test.values
        heldout_identifiers = tau_GT_test.index.values
    else:
        diagnoses_test = diagnoses.loc[diagnoses.index.isin(test_inds)]
        indices = diagnoses_test[diagnoses_test["dx"] == diagnosis].index
        y_true = tau_GT.loc[indices, "visual_read"].values
        y_pred_tau = tau_status_tau[tau_clusterthr][tau_intensitythr].loc[indices].values
        y_pred_anomaly = tau_status_anomaly[anomaly_clusterthr][anomaly_intensitythr].loc[indices].values
        heldout_identifiers = indices.values

    # Save held-out predictions for pooled out-of-fold evaluation
    y_true_allfolds.extend(y_true)
    y_pred_tau_allfolds.extend(y_pred_tau)
    y_pred_anomaly_allfolds.extend(y_pred_anomaly)
    identifier_allfolds.extend(heldout_identifiers)
    fold_allfolds.extend([fold] * len(y_true))

    report_tau = classification_report(y_true, y_pred_tau, labels=target_names, output_dict=True, zero_division=0)
    report_tau['accuracy'] = accuracy_score(y_true, y_pred_tau)
    reports_tau_allfolds.append(report_tau)
    report_anomaly = classification_report(y_true, y_pred_anomaly, labels=target_names, output_dict=True, zero_division=0)
    report_anomaly['accuracy'] = accuracy_score(y_true, y_pred_anomaly)
    reports_anomaly_allfolds.append(report_anomaly)

avg_report_tau, avg_report_anomaly = {}, {}
std_report_tau, std_report_anomaly = {}, {}

for key in reports_tau_allfolds[0].keys():
    if isinstance(reports_tau_allfolds[0][key], dict):  # e.g., '0', '1', 'macro avg', etc.
        avg_report_tau[key], avg_report_anomaly[key] = {}, {}
        std_report_tau[key], std_report_anomaly[key] = {}, {}
        for metric in reports_tau_allfolds[0][key].keys():
            avg_report_tau[key][metric] = np.mean([r[key][metric] for r in reports_tau_allfolds])
            avg_report_anomaly[key][metric] = np.mean([r[key][metric] for r in reports_anomaly_allfolds])
            std_report_tau[key][metric] = np.std([r[key][metric] for r in reports_tau_allfolds])
            std_report_anomaly[key][metric] = np.std([r[key][metric] for r in reports_anomaly_allfolds])
    else:  # e.g., 'accuracy' is a single float
        avg_report_tau[key] = np.mean([r[key] for r in reports_tau_allfolds])
        avg_report_anomaly[key] = np.mean([r[key] for r in reports_anomaly_allfolds])
        std_report_tau[key] = np.std([r[key] for r in reports_tau_allfolds])
        std_report_anomaly[key] = np.std([r[key] for r in reports_anomaly_allfolds])

# =========================================================
# Pooled out-of-fold performance
# =========================================================

y_true_allfolds = np.asarray(y_true_allfolds)
y_pred_tau_allfolds = np.asarray(y_pred_tau_allfolds)
y_pred_anomaly_allfolds = np.asarray(y_pred_anomaly_allfolds)

print("Total held-out observations:", len(y_true_allfolds))
print(
    "Visual-read negatives:",
    np.sum(y_true_allfolds == "tauNeg")
)
print(
    "Visual-read positives:",
    np.sum(y_true_allfolds == "tauPos")
)

def pooled_classification_metrics(y_true, y_pred, name):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=["tauNeg", "tauPos"]
    )

    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"TN: {tn}")
    print(f"FP: {fp}")
    print(f"FN: {fn}")
    print(f"TP: {tp}")
    print(f"Sensitivity:       {sensitivity:.3f}")
    print(f"Specificity:       {specificity:.3f}")
    print(f"Balanced accuracy: {balanced_acc:.3f}")
    print(f"Accuracy:          {accuracy:.3f}")

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": balanced_acc,
        "accuracy": accuracy,
    }


pooled_tau = pooled_classification_metrics(
    y_true_allfolds,
    y_pred_tau_allfolds,
    "Tau SUVR"
)

pooled_anomaly = pooled_classification_metrics(
    y_true_allfolds,
    y_pred_anomaly_allfolds,
    "Anomaly"
)

# =========================================================
# Save final cross-validation results
# =========================================================

results_dir = PROJECT_DIR / "visual_read_results"
results_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Pooled out-of-fold predictions
# ---------------------------------------------------------

oof_results = pd.DataFrame({
    "identifier": identifier_allfolds,
    "fold": fold_allfolds,
    "visual_read": y_true_allfolds,
    "pred_suvr": y_pred_tau_allfolds,
    "pred_anomaly": y_pred_anomaly_allfolds,
})

oof_results.to_csv(
    results_dir / f"{DATASET}_visual_read_oof_predictions.csv",
    index=False
)


# ---------------------------------------------------------
# 2. Selected thresholds for each fold
# ---------------------------------------------------------

threshold_results = pd.DataFrame({
    "fold": range(num_folds),
    "anomaly_intensity_threshold": [
        best_thresholds_anomaly["intensity"][fold]
        for fold in range(num_folds)
    ],
    "anomaly_cluster_threshold_mm3": [
        best_thresholds_anomaly["cluster_size"][fold]
        for fold in range(num_folds)
    ],
    "suvr_intensity_threshold": [
        best_thresholds_tau["intensity"][fold]
        for fold in range(num_folds)
    ],
    "suvr_cluster_threshold_mm3": [
        best_thresholds_tau["cluster_size"][fold]
        for fold in range(num_folds)
    ],
})

threshold_results.to_csv(
    results_dir / f"{DATASET}_visual_read_cv_thresholds.csv",
    index=False
)


# ---------------------------------------------------------
# 3. Pooled performance summary
# ---------------------------------------------------------

summary_results = pd.DataFrame([
    {
        "method": "SUVR",
        **pooled_tau
    },
    {
        "method": "Anomaly",
        **pooled_anomaly
    }
])

summary_results.to_csv(
    results_dir / f"{DATASET}_visual_read_pooled_metrics.csv",
    index=False
)


# ---------------------------------------------------------
# 4. Sample composition
# ---------------------------------------------------------

sample_summary = pd.DataFrame([{
    "DATASET": DATASET,
    "n_total": len(y_true_allfolds),
    "n_visual_read_negative": np.sum(
        np.asarray(y_true_allfolds) == "tauNeg"
    ),
    "n_visual_read_positive": np.sum(
        np.asarray(y_true_allfolds) == "tauPos"
    ),
}])

sample_summary.to_csv(
    results_dir / f"{DATASET}_visual_read_sample_summary.csv",
    index=False
)


print(f"Saved visual-read results to: {results_dir}")

In [ ]:
def display_report(avg_report, std_report):
    # Separate accuracy
    accuracy = avg_report.pop("accuracy")
    std_accuracy = std_report.pop("accuracy")

    # Convert to DataFrame
    avg_df = pd.DataFrame.from_dict(avg_report, orient="index")
    std_df = pd.DataFrame.from_dict(std_report, orient="index")

    # Format numbers
    formatted_avg = avg_df.applymap(lambda x: f"{x:.2f}")
    formatted_std = std_df.applymap(lambda x: f"{x:.3f}")
    combined = formatted_avg + " ± " + formatted_std

    # Add accuracy as a separate row
    combined.loc["accuracy"] = [f"{accuracy:.2f} ± {std_accuracy:.3f}"] + [""] * (combined.shape[1] - 1)

    # Display
    print(combined)
    print("\n")


print(f"Tau SUVR (averaged over {num_folds} folds):")
display_report(avg_report_tau, std_report_tau)
print(f"Anomaly map (averaged over {num_folds} folds):")
display_report(avg_report_anomaly, std_report_anomaly)